# Data Cleaning Pipeline

This notebook focuses on cleaning the raw heart attack dataset. The steps include handling missing values, standardizing data types, treating outliers, and saving a clean version for Exploratory Data Analysis (EDA).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 1. Load the Data

In [2]:
# Load raw dataset
df = pd.read_csv('../data/raw/heart_attack_dataset_raw.csv')
print(f"Original shape: {df.shape}")
df.head()

Original shape: (7000, 22)


,patient_id,age,gender,chest_pain_type,resting_bp,cholesterol,fasting_blood_sugar,resting_ecg,max_heart_rate,exercise_angina,oldpeak,st_slope,num_major_vessels,thalassemia,bmi,smoking_status,alcohol_consumption,physical_activity,family_history,diabetes,stress_level,heart_attack_risk
0,1,59,Male,Asymptomatic,105.0,154.0,0.0,ST-T Abnormality,186.0,1,1.0,Flat,1,Fixed Defect,23.5,Former,Heavy,Moderate,1,1,6.0,0
1,2,50,Male,Non-anginal Pain,102.0,180.0,0.0,Normal,183.0,0,2.9,Flat,0,Reversible Defect,19.3,NaN,Moderate,High,0,0,8.0,0
2,3,61,Male,Atypical Angina,117.0,213.0,1.0,Normal,161.0,0,0.0,Flat,1,Fixed Defect,30.5,Never,Moderate,Low,1,1,2.0,1
3,4,73,Female,Atypical Angina,121.0,208.0,1.0,Normal,150.0,0,1.0,Up,0,Normal,28.6,Former,Moderate,Moderate,0,0,4.0,0
4,5,49,Male,Non-anginal Pain,106.0,157.0,0.0,Normal,185.0,0,0.3,Flat,0,Reversible Defect,21.4,Former,Moderate,Moderate,0,0,NaN,0


## 2. Drop Duplicates

In [3]:
# Check for and drop duplicates
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Dropped {initial_rows - df.shape[0]} duplicate rows.")

Dropped 0 duplicate rows.


## 3. Handle Missing Values

Let's identify the missing values first.

In [4]:
missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

smoking_status         505
alcohol_consumption    405
cholesterol            398
oldpeak                350
physical_activity      346
bmi                    330
resting_bp             280
max_heart_rate         279
stress_level           268
resting_ecg            199
fasting_blood_sugar    196
dtype: int64

### 3.1 Impute Numerical Features
We will impute missing values in continuous numerical features using the median, as it is robust to outliers.

In [5]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
num_cols = num_cols.drop(['patient_id', 'heart_attack_risk'], errors='ignore') # Exclude ID and target if present

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Imputed {col} with median: {median_val}")

Imputed resting_bp with median: 116.0
Imputed cholesterol with median: 184.0
Imputed fasting_blood_sugar with median: 0.0
Imputed max_heart_rate with median: 174.0
Imputed oldpeak with median: 1.0
Imputed bmi with median: 27.1
Imputed stress_level with median: 5.0


### 3.2 Impute Categorical Features
For categorical features, we will impute missing values using the mode (most frequent value) or assign them to an 'Unknown' category.

In [6]:
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Imputed {col} with mode: {mode_val}")

Imputed resting_ecg with mode: Normal
Imputed smoking_status with mode: Never
Imputed alcohol_consumption with mode: Non-drinker
Imputed physical_activity with mode: Moderate


## 4. Standardize Data Types

Ensure categorical features are represented as `category` type for memory efficiency and compatibility with certain models.

In [7]:
# Convert object columns to category
for col in cat_cols:
    df[col] = df[col].astype('category')
    
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   patient_id           7000 non-null   int64   
 1   age                  7000 non-null   int64   
 2   gender               7000 non-null   category
 3   chest_pain_type      7000 non-null   category
 4   resting_bp           7000 non-null   float64 
 5   cholesterol          7000 non-null   float64 
 6   fasting_blood_sugar  7000 non-null   float64 
 7   resting_ecg          7000 non-null   category
 8   max_heart_rate       7000 non-null   float64 
 9   exercise_angina      7000 non-null   int64   
 10  oldpeak              7000 non-null   float64 
 11  st_slope             7000 non-null   category
 12  num_major_vessels    7000 non-null   int64   
 13  thalassemia          7000 non-null   category
 14  bmi                  7000 non-null   float64 
 15  smoking_status       

## 5. Handle Outliers

We can identify and cap outliers in numerical columns using the Interquartile Range (IQR) method.

In [8]:
def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count outliers
    outliers_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
    
    # Cap outliers
    dataframe[column] = np.where(dataframe[column] < lower_bound, lower_bound, dataframe[column])
    dataframe[column] = np.where(dataframe[column] > upper_bound, upper_bound, dataframe[column])
    
    return outliers_count

outlier_columns = ['resting_bp', 'cholesterol', 'max_heart_rate', 'bmi']
for col in outlier_columns:
    count = cap_outliers_iqr(df, col)
    print(f"Capped {count} outliers in {col}")

Capped 69 outliers in resting_bp
Capped 68 outliers in cholesterol
Capped 24 outliers in max_heart_rate
Capped 37 outliers in bmi


## 6. Save Cleaned Dataset

In [9]:
# Check final missing values count
assert df.isnull().sum().sum() == 0, "There are still missing values!"

# Save to interim data directory
import os
os.makedirs('../data/interim', exist_ok=True)
cleaned_data_path = '../data/interim/heart_attack_dataset_cleaned.csv'
df.to_csv(cleaned_data_path, index=False)
print(f"Cleaned dataset saved to: {cleaned_data_path}")

Cleaned dataset saved to: ../data/interim/heart_attack_dataset_cleaned.csv
